In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path (so imports like src.data work)
project_root = Path().cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import logger

In [ ]:
from src import DATA_BASE_DIR
import yaml

In [ ]:
def generate_data_txt_for_sequences(base_dirs_with_sequences, output_file="val.txt", max_entries=None):
    """
    Generate txt listing images (with labels) only for specified sequences.
    
    Parameters
    ----------
    base_dirs_with_sequences : list of (str, list of str)
        Example: [(base_dir1, ["seq1","seq2"]), (base_dir2, ["seqA","seqB"]), ...]
    output_file : str
        Output txt filename
    max_entries : int, optional
        Limit number of entries, if one wants to run on a subset

    Returns
    -------
    nr_entries: int
        Number of entries
    output_file: str
        Path to output_file
    
    Raises
    ------
    FileNotFoundError
    """
    valid_entries = []
    print(base_dirs_with_sequences)
    for base_dir, sequences in base_dirs_with_sequences:
        labels_root = os.path.join(base_dir, "labels")
        images_root = os.path.join(base_dir, "images")
        
        for seq in sequences:
            # Check, if the label and image directories exist.
            labels_seq_dir = os.path.join(labels_root, seq)
            images_seq_dir = os.path.join(images_root, seq)
    
            if not os.path.isdir(labels_seq_dir) or not os.path.isdir(images_seq_dir):
                raise FileNotFoundError(f"Missing image or label directory for {seq} in {base_dir}")
            logger.info(f"Found {labels_seq_dir} and {images_seq_dir}")


            for img_file in sorted(os.listdir(images_seq_dir)):
                if not img_file.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                    
                image_path = os.path.join(images_seq_dir, img_file)
                label_path = os.path.join(labels_seq_dir, os.path.splitext(img_file)[0] + ".txt")
            
                if not os.path.exists(label_path):
                    # Create empty txt for background images
                    with open(label_path, "x") as f:
                        pass

                valid_entries.append(os.path.abspath(image_path))
                if max_entries is not None and len(valid_entries)>=max_entries:
                    break

    nr_entries = len(valid_entries)
    if nr_entries == 0:
        raise FileNotFoundError(f"No entries for {output_file}.")    
    
    with open(output_file, "w") as f:
        f.write('\n'.join(valid_entries))
    
    print(f"{output_file} created with {nr_entries} entries at {output_file}")
    return nr_entries, output_file

# PMOF Val

In [ ]:
CEPDOF_BASE_DIR = '/home/stella/computer_vision/fisheye_data'
CEPDOF_AUG_BASE_DIR = '/home/stella/computer_vision/fisheye_data/augmentation_study'
PMOF_BASE_DIR = str(DATA_BASE_DIR)
PMOF_AUG_BASE_DIR = '/home/stella/computer_vision/PMOF_aug'


In [ ]:
datasets_train = {
    "C" : (CEPDOF_BASE_DIR, ["Lunch1", "Lunch2", "Lunch3", "All_off", "High_activity", "IRfilter", "IRill", "Edge_cases"]), #CEPDOF
    "Ca18" : (CEPDOF_AUG_BASE_DIR, ["Lunch1_a18", "Lunch2_a18", "Lunch3_a18", "All_off_a18", "High_activity_a18", "IRfilter_a18", "IRill_a18", "Edge_cases_a18"]), #Augmented CEPDOF
 
    "P" : (PMOF_BASE_DIR, [f"rec{i}" for i in range(1, 31)]), #PMOF_passenger
    "Pa18" : (PMOF_AUG_BASE_DIR, [f"rec{i}_a18" for i in range(1, 26)]), #PMOF augmented
}

file_name = ""
all_subsets = []
for name, subsets in datasets_train.items():
    file_name += name
    all_subsets.append((subsets))
print(file_name)
print(all_subsets)
generate_data_txt_for_sequences(all_subsets, output_file=f"dataset_txt/{file_name}-P.txt")


# Create the matching yaml file
train_txt =  Path().cwd().resolve() / f"dataset_txt/{file_name}-P.txt"
relative_train_txt = str(train_txt.relative_to(Path().home()))

val_txt = DATA_BASE_DIR/ "val.txt" ### ISSUE: the val.txt is generated & does not have the correct path
relative_val_txt = str(val_txt.relative_to(Path().home()))

home_path = str(Path().home())


yaml_text = dict(
    path = home_path,
    train = relative_train_txt,
    val = relative_val_txt,
    names = {0: 'person'}
)

with open(f'yaml/{file_name}-P.yaml', 'w') as outfile:
    yaml.dump(yaml_text, outfile, default_flow_style=False, sort_keys=False)

